# Notebook 02: Component Selection Experiments

**Google Colab Version** — Mounts Google Drive and installs dependencies.

## 0. Google Colab Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install pandas numpy matplotlib seaborn openpyxl tqdm python-dotenv -q
!pip install langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu rank-bm25 chromadb qdrant-client groq ragas pyarrow -q

COLAB_PROJECT_ROOT = '/content/drive/MyDrive/MedQA-Thesis'
print(f'Project root: {COLAB_PROJECT_ROOT}')

## 1. Setup & Imports

In [ ]:
import json
import os
import re
import time
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from tqdm import tqdm

# LangChain text splitters
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

# Embeddings
from sentence_transformers import SentenceTransformer

# Retrieval
import faiss
from rank_bm25 import BM25Okapi

# LLM
from groq import Groq

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path(COLAB_PROJECT_ROOT)  # Colab: uses Google Drive path
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INDICES = PROJECT_ROOT / "data" / "indices"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
RESULTS_DIR = PROJECT_ROOT / "results" / "experiments"
EXCEL_PATH = EXPERIMENTS_DIR / "ThesisExperimentData.xlsx"

DATA_INDICES.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Plot settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Excel path: {EXCEL_PATH.resolve()}")

In [ ]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================
# Option 1: Set directly here (not recommended for version control)
GROQ_API_KEY = "your-groq-api-key-here"

# Option 2: Load from environment variable
# import os
# GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-api-key-here")

# Option 3: Load from config file
# import sys; sys.path.insert(0, str(PROJECT_ROOT / "config"))
# from config import GROQ_API_KEY

if GROQ_API_KEY == "your-groq-api-key-here":
    print("WARNING: Please set your Groq API key above before running LLM experiments.")
else:
    print("Groq API key loaded successfully.")

## 2. Load Preprocessed Data

In [ ]:
# Load preprocessed data from Notebook 01
df_dev = pd.read_parquet(DATA_PROCESSED / "medqa_dev.parquet")

with open(DATA_PROCESSED / "textbook_corpus.json", "r", encoding="utf-8") as f:
    textbook_corpus = json.load(f)

# Sample 50 questions from dev set for fast experiments
SAMPLE_SIZE = 50
df_sample = df_dev.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Dev set: {len(df_dev):,} questions")
print(f"Sample for experiments: {len(df_sample)} questions")
print(f"Textbooks loaded: {len(textbook_corpus)} books")
print(f"Total corpus characters: {sum(len(b['text']) for b in textbook_corpus):,}")

## 3. Define Chunking Strategies

In [ ]:
def combine_corpus_text(textbook_corpus):
    """Combine all textbook texts with book name markers."""
    all_texts = []
    for book in textbook_corpus:
        all_texts.append(book["text"])
    return all_texts

def chunk_fixed_size(texts, chunk_size=200, book_names=None):
    """Fixed-size chunking without overlap."""
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=0,
        length_function=len,
    )
    chunks = []
    for i, text in enumerate(texts):
        book = book_names[i] if book_names else f"book_{i}"
        for chunk in splitter.split_text(text):
            if chunk.strip():
                chunks.append({"text": chunk.strip(), "source": book})
    return chunks

def chunk_overlapping(texts, chunk_size=512, chunk_overlap=50, book_names=None):
    """Overlapping chunking."""
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = []
    for i, text in enumerate(texts):
        book = book_names[i] if book_names else f"book_{i}"
        for chunk in splitter.split_text(text):
            if chunk.strip():
                chunks.append({"text": chunk.strip(), "source": book})
    return chunks

def chunk_recursive(texts, chunk_size=512, chunk_overlap=50, book_names=None):
    """Recursive chunking using natural boundaries."""
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " "],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = []
    for i, text in enumerate(texts):
        book = book_names[i] if book_names else f"book_{i}"
        for chunk in splitter.split_text(text):
            if chunk.strip():
                chunks.append({"text": chunk.strip(), "source": book})
    return chunks

def chunk_semantic_passage(texts, chunk_size=200, book_names=None):
    """Semantic passage chunking — sentence-boundary aware fixed-size."""
    splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", "! ", "? ", " "],
        chunk_size=chunk_size,
        chunk_overlap=20,
        length_function=len,
    )
    chunks = []
    for i, text in enumerate(texts):
        book = book_names[i] if book_names else f"book_{i}"
        for chunk in splitter.split_text(text):
            if chunk.strip():
                chunks.append({"text": chunk.strip(), "source": book})
    return chunks

def chunk_sentence_window(texts, window_size=4, overlap=1, book_names=None):
    """Sentence window chunking — groups of N sentences with overlap."""
    import re
    chunks = []
    for i, text in enumerate(texts):
        book = book_names[i] if book_names else f"book_{i}"
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        step = window_size - overlap
        if step < 1:
            step = 1
        for start in range(0, len(sentences), step):
            window = sentences[start:start + window_size]
            chunk_text = " ".join(window)
            if chunk_text.strip():
                chunks.append({"text": chunk_text.strip(), "source": book})
    return chunks

# Define all chunking strategy configs
CHUNKING_CONFIGS = {
    "Fixed-Size (200 tokens)": lambda texts, bn: chunk_fixed_size(texts, chunk_size=200, book_names=bn),
    "Fixed-Size (512 tokens)": lambda texts, bn: chunk_fixed_size(texts, chunk_size=512, book_names=bn),
    "Overlapping (512 tokens, 50 overlap)": lambda texts, bn: chunk_overlapping(texts, 512, 50, bn),
    "Overlapping (512 tokens, 100 overlap)": lambda texts, bn: chunk_overlapping(texts, 512, 100, bn),
    "Recursive (512 tokens)": lambda texts, bn: chunk_recursive(texts, 512, 50, bn),
    "Semantic Passage (200 tokens)": lambda texts, bn: chunk_semantic_passage(texts, 200, bn),
    "Sentence Window (4 sentences, overlap=1)": lambda texts, bn: chunk_sentence_window(texts, 4, 1, bn),
}

print(f"Defined {len(CHUNKING_CONFIGS)} chunking strategies")

## 4. Define Evaluation Metrics

In [ ]:
def compute_exact_match(predictions, ground_truths):
    """Exact match accuracy: predicted option letter matches ground truth."""
    correct = sum(1 for p, g in zip(predictions, ground_truths) if p.strip().upper() == g.strip().upper())
    return correct / len(predictions) if predictions else 0

def compute_token_f1(prediction, ground_truth):
    """Token-level F1 between predicted answer text and ground truth text."""
    pred_tokens = set(prediction.lower().split())
    gt_tokens = set(ground_truth.lower().split())
    
    if not pred_tokens or not gt_tokens:
        return 0.0
    
    common = pred_tokens & gt_tokens
    if not common:
        return 0.0
    
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_retrieval_recall_at_k(retrieved_chunks_list, ground_truth_answers, k):
    """Retrieval Recall@k: fraction of questions where at least one of top-k
    retrieved chunks contains a token from the correct answer."""
    hits = 0
    for chunks, answer in zip(retrieved_chunks_list, ground_truth_answers):
        top_k = chunks[:k]
        answer_tokens = set(answer.lower().split())
        for chunk in top_k:
            chunk_tokens = set(chunk.lower().split())
            if answer_tokens & chunk_tokens:
                hits += 1
                break
    return hits / len(ground_truth_answers) if ground_truth_answers else 0

def compute_mrr(retrieved_chunks_list, ground_truth_answers):
    """Mean Reciprocal Rank: average of 1/rank of first relevant chunk."""
    rr_sum = 0
    for chunks, answer in zip(retrieved_chunks_list, ground_truth_answers):
        answer_tokens = set(answer.lower().split())
        for rank, chunk in enumerate(chunks, 1):
            chunk_tokens = set(chunk.lower().split())
            if answer_tokens & chunk_tokens:
                rr_sum += 1.0 / rank
                break
    return rr_sum / len(ground_truth_answers) if ground_truth_answers else 0

def compute_precision_recall_f1(predictions, ground_truths):
    """Compute macro precision, recall, F1 for multi-class (A/B/C/D)."""
    labels = ["A", "B", "C", "D"]
    tp = {l: 0 for l in labels}
    fp = {l: 0 for l in labels}
    fn = {l: 0 for l in labels}
    
    for pred, gt in zip(predictions, ground_truths):
        pred = pred.strip().upper()
        gt = gt.strip().upper()
        for l in labels:
            if pred == l and gt == l:
                tp[l] += 1
            elif pred == l and gt != l:
                fp[l] += 1
            elif pred != l and gt == l:
                fn[l] += 1
    
    precisions, recalls, f1s = [], [], []
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if (tp[l] + fp[l]) > 0 else 0
        r = tp[l] / (tp[l] + fn[l]) if (tp[l] + fn[l]) > 0 else 0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f)
    
    return np.mean(precisions), np.mean(recalls), np.mean(f1s)

def parse_answer_letter(response_text):
    """Extract the answer letter (A/B/C/D) from LLM response."""
    text = response_text.strip()
    
    # Pattern 1: "Answer: A" or "Answer: B"
    match = re.search(r'Answer\s*:\s*([A-D])', text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # Pattern 2: Starts with just the letter
    match = re.match(r'^([A-D])[.\)\s]', text)
    if match:
        return match.group(1).upper()
    
    # Pattern 3: "The answer is A"
    match = re.search(r'the\s+answer\s+is\s+([A-D])', text, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # Pattern 4: Look for first standalone A/B/C/D
    match = re.search(r'\b([A-D])\b', text)
    if match:
        return match.group(1).upper()
    
    return "X"  # Could not parse

print("Evaluation metrics defined.")

## 5. Define RAG Pipeline Components

In [ ]:
# ============================================================
# PROMPT TEMPLATE (shared across ALL experiments)
# ============================================================
PROMPT_TEMPLATE = """You are a medical expert answering USMLE-style clinical questions.
Use ONLY the provided evidence passages to answer. If the evidence is insufficient, state that rather than guessing.

Evidence Passages:
{context}

Question: {question}

Options:
A) {option_a}
B) {option_b}
C) {option_c}
D) {option_d}

Instructions:
1. Analyze the evidence passages relevant to the question.
2. Select the single best answer (A, B, C, or D).
3. Provide a brief explanation citing specific evidence passages.

Answer format:
Answer: [letter]
Explanation: [your reasoning with evidence citations]"""


def build_prompt(question_row, retrieved_texts, top_k=5):
    """Build a prompt from a question and retrieved text chunks."""
    context = "\n\n".join([f"[Passage {i+1}]: {t}" for i, t in enumerate(retrieved_texts[:top_k])])
    return PROMPT_TEMPLATE.format(
        context=context,
        question=question_row["question"],
        option_a=question_row["option_a"],
        option_b=question_row["option_b"],
        option_c=question_row["option_c"],
        option_d=question_row["option_d"],
    )


# ============================================================
# EMBEDDING FUNCTIONS
# ============================================================
def load_embedding_model(model_name):
    """Load a sentence-transformer embedding model."""
    print(f"  Loading embedding model: {model_name}...")
    model = SentenceTransformer(model_name)
    return model

def embed_texts(model, texts, batch_size=64):
    """Embed a list of texts using a sentence-transformer model."""
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)
    return embeddings.astype("float32")


# ============================================================
# RETRIEVAL FUNCTIONS
# ============================================================
def build_faiss_index(embeddings):
    """Build a FAISS IndexFlatIP (inner product = cosine for normalized vectors)."""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def faiss_retrieve(index, query_embedding, top_k=5):
    """Retrieve top-k chunks from FAISS index."""
    scores, indices = index.search(query_embedding.reshape(1, -1), top_k)
    return indices[0].tolist(), scores[0].tolist()

def build_bm25_index(chunk_texts):
    """Build a BM25 index from tokenized chunk texts."""
    tokenized = [text.lower().split() for text in chunk_texts]
    return BM25Okapi(tokenized)

def bm25_retrieve(bm25_index, query, top_k=5):
    """Retrieve top-k chunks from BM25 index."""
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices.tolist(), scores[top_indices].tolist()

def rrf_fuse(faiss_indices, bm25_indices, k=60):
    """Reciprocal Rank Fusion of two ranked lists."""
    scores = {}
    for rank, idx in enumerate(faiss_indices):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in sorted_items]


# ============================================================
# LLM FUNCTION
# ============================================================
def call_llm(client, prompt, model_name="llama-3.3-70b-versatile", temperature=0):
    """Call Groq API with rate limiting and retry."""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=500,
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait_time = 10 * (attempt + 1)
                print(f"    Rate limited. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"    LLM error: {e}")
                return "Error: Could not generate response"
    return "Error: Rate limit exceeded after retries"

# Map model display names to Groq API model IDs
LLM_MODEL_MAP = {
    "LLaMA 3.3 70B": "llama-3.3-70b-versatile",
    "LLaMA 3.1 8B": "llama-3.1-8b-instant",
    "Mixtral 8x7B": "mixtral-8x7b-32768",
}

print("RAG pipeline components defined.")

## 6. Experiment Runner

The core function that runs a single experiment: chunk → embed → index → retrieve → generate → evaluate.

In [ ]:
def run_experiment(
    df_questions,
    textbook_corpus,
    chunking_name,
    embedding_model_name,
    vectordb_name,
    llm_model_name,
    rag_type="Naive RAG",
    top_k=5,
    groq_client=None,
    precomputed=None,
):
    """
    Run a single experiment and return all metrics.
    
    Args:
        df_questions: DataFrame with sample questions
        textbook_corpus: list of {book_name, text} dicts
        chunking_name: key from CHUNKING_CONFIGS
        embedding_model_name: HuggingFace model name
        vectordb_name: "FAISS", "ChromaDB", or "Qdrant"
        llm_model_name: display name from LLM_MODEL_MAP
        rag_type: "Naive RAG", "Sparse RAG", "Hybrid RAG", or "Multi-Hop RAG"
        top_k: number of chunks to retrieve
        groq_client: Groq client instance
        precomputed: dict of precomputed chunks/embeddings/indices to reuse
    
    Returns:
        dict of all metrics
    """
    print(f"\n{'='*60}")
    print(f"  RAG: {rag_type} | Chunking: {chunking_name}")
    print(f"  Embedding: {embedding_model_name} | VectorDB: {vectordb_name} | LLM: {llm_model_name}")
    print(f"{'='*60}")
    
    # --- Step 1: Chunking ---
    if precomputed and "chunks" in precomputed:
        chunks = precomputed["chunks"]
        print(f"  [1/6] Chunks: reusing {len(chunks):,} precomputed chunks")
    else:
        texts = [b["text"] for b in textbook_corpus]
        book_names = [b["book_name"] for b in textbook_corpus]
        chunk_fn = CHUNKING_CONFIGS[chunking_name]
        chunks = chunk_fn(texts, book_names)
        print(f"  [1/6] Chunking: {len(chunks):,} chunks created")
    
    chunk_texts = [c["text"] for c in chunks]
    
    # --- Step 2: Embedding ---
    if precomputed and "embeddings" in precomputed and "emb_model" in precomputed:
        embeddings = precomputed["embeddings"]
        emb_model = precomputed["emb_model"]
        print(f"  [2/6] Embeddings: reusing precomputed ({embeddings.shape})")
    else:
        emb_model = load_embedding_model(embedding_model_name)
        print(f"  [2/6] Embedding {len(chunk_texts):,} chunks...")
        embeddings = embed_texts(emb_model, chunk_texts)
        print(f"         Embeddings shape: {embeddings.shape}")
    
    # --- Step 3: Build index ---
    if rag_type == "Sparse RAG":
        # BM25 only — no embedding index needed
        bm25 = build_bm25_index(chunk_texts)
        print(f"  [3/6] BM25 index built")
    elif rag_type == "Hybrid RAG":
        # Both FAISS and BM25
        faiss_index = build_faiss_index(embeddings)
        bm25 = build_bm25_index(chunk_texts)
        print(f"  [3/6] FAISS + BM25 indices built")
    else:
        # Naive RAG and Multi-Hop RAG use FAISS
        if vectordb_name == "FAISS":
            faiss_index = build_faiss_index(embeddings)
            print(f"  [3/6] FAISS index built ({faiss_index.ntotal} vectors)")
        elif vectordb_name == "ChromaDB":
            import chromadb
            chroma_client = chromadb.Client()
            # Delete collection if exists
            try:
                chroma_client.delete_collection("medqa_exp")
            except:
                pass
            collection = chroma_client.create_collection(
                name="medqa_exp",
                metadata={"hnsw:space": "cosine"}
            )
            # Add in batches (ChromaDB has limits)
            batch_size = 5000
            for i in range(0, len(chunk_texts), batch_size):
                batch_end = min(i + batch_size, len(chunk_texts))
                collection.add(
                    ids=[str(j) for j in range(i, batch_end)],
                    embeddings=embeddings[i:batch_end].tolist(),
                    documents=chunk_texts[i:batch_end],
                )
            print(f"  [3/6] ChromaDB collection built ({collection.count()} vectors)")
        elif vectordb_name == "Qdrant":
            from qdrant_client import QdrantClient
            from qdrant_client.models import VectorParams, Distance, PointStruct
            qdrant = QdrantClient(":memory:")
            dim = embeddings.shape[1]
            qdrant.recreate_collection(
                collection_name="medqa_exp",
                vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
            )
            # Add in batches
            batch_size = 1000
            for i in range(0, len(chunk_texts), batch_size):
                batch_end = min(i + batch_size, len(chunk_texts))
                points = [
                    PointStruct(id=j, vector=embeddings[j].tolist(), payload={"text": chunk_texts[j]})
                    for j in range(i, batch_end)
                ]
                qdrant.upsert(collection_name="medqa_exp", points=points)
            print(f"  [3/6] Qdrant collection built ({len(chunk_texts)} vectors)")
    
    # --- Step 4: Retrieve + Generate ---
    print(f"  [4/6] Running RAG pipeline on {len(df_questions)} questions...")
    predictions = []
    ground_truths = []
    all_retrieved_texts = []
    all_responses = []
    
    llm_api_model = LLM_MODEL_MAP.get(llm_model_name, llm_model_name)
    
    for idx, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="  Generating"):
        question = row["question"]
        gt_letter = row["answer_idx"]
        gt_answer = row["answer"]
        
        # Retrieve
        if rag_type == "Sparse RAG":
            ret_indices, _ = bm25_retrieve(bm25, question, top_k=top_k)
            retrieved_texts = [chunk_texts[i] for i in ret_indices]
        
        elif rag_type == "Hybrid RAG":
            query_emb = embed_texts(emb_model, [question])
            faiss_idx, _ = faiss_retrieve(faiss_index, query_emb[0], top_k=top_k * 2)
            bm25_idx, _ = bm25_retrieve(bm25, question, top_k=top_k * 2)
            fused_indices = rrf_fuse(faiss_idx, bm25_idx)[:top_k]
            retrieved_texts = [chunk_texts[i] for i in fused_indices]
        
        elif rag_type == "Multi-Hop RAG":
            # Simplified multi-hop: 2 hops
            query_emb = embed_texts(emb_model, [question])
            hop1_idx, _ = faiss_retrieve(faiss_index, query_emb[0], top_k=3)
            hop1_texts = [chunk_texts[i] for i in hop1_idx]
            
            # Refine query with hop1 context
            refined_query = question + " " + " ".join(hop1_texts[:1])[:200]
            refined_emb = embed_texts(emb_model, [refined_query])
            hop2_idx, _ = faiss_retrieve(faiss_index, refined_emb[0], top_k=top_k)
            
            # Combine unique chunks from both hops
            seen = set()
            retrieved_texts = []
            for i in hop1_idx + hop2_idx:
                if i not in seen:
                    seen.add(i)
                    retrieved_texts.append(chunk_texts[i])
            retrieved_texts = retrieved_texts[:top_k]
        
        else:
            # Naive RAG
            query_emb = embed_texts(emb_model, [question])
            if vectordb_name == "FAISS":
                ret_indices, _ = faiss_retrieve(faiss_index, query_emb[0], top_k=top_k)
                retrieved_texts = [chunk_texts[i] for i in ret_indices]
            elif vectordb_name == "ChromaDB":
                results = collection.query(query_embeddings=query_emb[0].tolist(), n_results=top_k)
                retrieved_texts = results["documents"][0]
            elif vectordb_name == "Qdrant":
                results = qdrant.search(
                    collection_name="medqa_exp",
                    query_vector=query_emb[0].tolist(),
                    limit=top_k,
                )
                retrieved_texts = [hit.payload["text"] for hit in results]
        
        all_retrieved_texts.append(retrieved_texts)
        
        # Generate
        prompt = build_prompt(row, retrieved_texts, top_k=top_k)
        
        if groq_client:
            response = call_llm(groq_client, prompt, model_name=llm_api_model)
            time.sleep(1)  # Rate limiting
        else:
            response = "Error: No LLM client provided"
        
        all_responses.append(response)
        pred_letter = parse_answer_letter(response)
        predictions.append(pred_letter)
        ground_truths.append(gt_letter)
    
    # --- Step 5: Compute Metrics ---
    print(f"  [5/6] Computing metrics...")
    
    accuracy = compute_exact_match(predictions, ground_truths)
    precision, recall, f1 = compute_precision_recall_f1(predictions, ground_truths)
    exact_match = accuracy  # Same as accuracy for MCQ
    
    # Token F1 (compare full answer text)
    gt_answers = df_questions["answer"].tolist()
    token_f1_scores = [compute_token_f1(resp, ans) for resp, ans in zip(all_responses, gt_answers)]
    token_f1 = np.mean(token_f1_scores)
    
    # Retrieval metrics
    recall_at_3 = compute_retrieval_recall_at_k(all_retrieved_texts, gt_answers, k=3)
    recall_at_5 = compute_retrieval_recall_at_k(all_retrieved_texts, gt_answers, k=5)
    recall_at_10 = compute_retrieval_recall_at_k(all_retrieved_texts, gt_answers, k=min(10, top_k))
    mrr = compute_mrr(all_retrieved_texts, gt_answers)
    
    metrics = {
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "Exact_Match": round(exact_match, 4),
        "Token_F1": round(token_f1, 4),
        "Recall@3": round(recall_at_3, 4),
        "Recall@5": round(recall_at_5, 4),
        "Recall@10": round(recall_at_10, 4),
        "MRR": round(mrr, 4),
        "RAGAS_Faithfulness": None,  # Computed separately if needed
        "RAGAS_Answer_Correctness": None,
        "RAGAS_Context_Precision": None,
        "RAGAS_Context_Recall": None,
        "RAGAS_Answer_Relevancy": None,
    }
    
    print(f"  [6/6] Results: Accuracy={accuracy:.4f} | F1={f1:.4f} | MRR={mrr:.4f}")
    
    return metrics, {
        "predictions": predictions,
        "ground_truths": ground_truths,
        "responses": all_responses,
        "retrieved_texts": all_retrieved_texts,
        "chunks": chunks,
        "embeddings": embeddings,
        "emb_model": emb_model,
    }

print("Experiment runner defined.")

## 7. Excel Writer Utility

In [ ]:
def write_metrics_to_excel(excel_path, sn, metrics):
    """Write experiment metrics to the Excel spreadsheet at the row matching SN."""
    wb = openpyxl.load_workbook(excel_path)
    ws = wb["Sheet1"]
    
    # Find the row with matching SN
    target_row = None
    for row in range(2, ws.max_row + 1):
        if ws.cell(row=row, column=1).value == sn:
            target_row = row
            break
    
    if target_row is None:
        print(f"  WARNING: SN {sn} not found in Excel sheet!")
        return
    
    # Metric column mapping (column G=7 onwards)
    metric_cols = {
        "Accuracy": 7,
        "Precision": 8,
        "Recall": 9,
        "F1": 10,
        "Exact_Match": 11,
        "Token_F1": 12,
        "Recall@3": 13,
        "Recall@5": 14,
        "Recall@10": 15,
        "MRR": 16,
        "RAGAS_Faithfulness": 17,
        "RAGAS_Answer_Correctness": 18,
        "RAGAS_Context_Precision": 19,
        "RAGAS_Context_Recall": 20,
        "RAGAS_Answer_Relevancy": 21,
    }
    
    for metric_name, col_idx in metric_cols.items():
        value = metrics.get(metric_name)
        if value is not None:
            ws.cell(row=target_row, column=col_idx, value=value)
    
    wb.save(excel_path)
    print(f"  Metrics written to Excel row {target_row} (SN={sn})")

# Store all experiment results in memory
all_results = {}

print("Excel writer utility defined.")

In [ ]:
# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY != "your-groq-api-key-here" else None

if groq_client:
    print("Groq client initialized.")
else:
    print("WARNING: Groq client NOT initialized. Set your API key in Cell 2.")

## 8. Phase 1: Chunking Strategy Comparison (SN 1-7)

**Variable**: 7 chunking strategies  
**Fixed**: Embedding=all-MiniLM-L6-v2, VectorDB=FAISS, LLM=LLaMA 3.3 70B, RAG=Naive

In [ ]:
# Phase 1: Chunking Strategy Comparison
phase1_configs = [
    (1, "Fixed-Size (200 tokens)"),
    (2, "Fixed-Size (512 tokens)"),
    (3, "Overlapping (512 tokens, 50 overlap)"),
    (4, "Overlapping (512 tokens, 100 overlap)"),
    (5, "Recursive (512 tokens)"),
    (6, "Semantic Passage (200 tokens)"),
    (7, "Sentence Window (4 sentences, overlap=1)"),
]

P1_EMBEDDING = "all-MiniLM-L6-v2"
P1_VECTORDB = "FAISS"
P1_LLM = "LLaMA 3.3 70B"

print("Phase 1: Testing 7 chunking strategies")
print(f"Fixed: Embedding={P1_EMBEDDING}, VectorDB={P1_VECTORDB}, LLM={P1_LLM}\n")

for sn, chunking_name in phase1_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=chunking_name,
        embedding_model_name=P1_EMBEDDING,
        vectordb_name=P1_VECTORDB,
        llm_model_name=P1_LLM,
        rag_type="Naive RAG",
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

print("\nPhase 1 complete!")

In [ ]:
# Phase 1 Results Visualization
p1_data = []
for sn, chunking_name in phase1_configs:
    if sn in all_results:
        row = {"SN": sn, "Chunking": chunking_name}
        row.update(all_results[sn]["metrics"])
        p1_data.append(row)

df_p1 = pd.DataFrame(p1_data)
print("Phase 1 Results:")
print(df_p1[["SN", "Chunking", "Accuracy", "F1", "Recall@5", "MRR"]].to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ["Accuracy", "F1", "MRR"]):
    bars = ax.barh(df_p1["Chunking"], df_p1[metric], color=plt.cm.Set2(range(len(df_p1))))
    ax.set_title(f"Phase 1: {metric}", fontsize=13)
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, df_p1[metric]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9)

plt.suptitle("Phase 1: Chunking Strategy Comparison", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "phase1_chunking_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

# Select best chunking
best_p1_idx = df_p1["Accuracy"].idxmax()
BEST_CHUNKING = df_p1.loc[best_p1_idx, "Chunking"]
print(f"\n>>> BEST CHUNKING: {BEST_CHUNKING} (Accuracy={df_p1.loc[best_p1_idx, 'Accuracy']:.4f})")

## 9. Phase 2: Embedding Model Comparison (SN 8-10)

**Variable**: 3 embedding models  
**Fixed**: Chunking=best from Phase 1, VectorDB=FAISS, LLM=LLaMA 3.3 70B, RAG=Naive

In [ ]:
# Phase 2: Embedding Model Comparison
phase2_configs = [
    (8,  "BAAI/bge-large-en-v1.5"),
    (9,  "ncbi/MedCPT-Query-Encoder"),
    (10, "all-MiniLM-L6-v2"),
]

print(f"Phase 2: Testing 3 embedding models")
print(f"Fixed: Chunking={BEST_CHUNKING}, VectorDB=FAISS, LLM=LLaMA 3.3 70B\n")

for sn, emb_name in phase2_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=BEST_CHUNKING,
        embedding_model_name=emb_name,
        vectordb_name="FAISS",
        llm_model_name="LLaMA 3.3 70B",
        rag_type="Naive RAG",
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

# Visualize & select best
p2_data = [{"SN": sn, "Embedding": emb, **all_results[sn]["metrics"]} for sn, emb in phase2_configs if sn in all_results]
df_p2 = pd.DataFrame(p2_data)
print("\nPhase 2 Results:")
print(df_p2[["SN", "Embedding", "Accuracy", "F1", "Recall@5", "MRR"]].to_string(index=False))

best_p2_idx = df_p2["Accuracy"].idxmax()
BEST_EMBEDDING = df_p2.loc[best_p2_idx, "Embedding"]
print(f"\n>>> BEST EMBEDDING: {BEST_EMBEDDING} (Accuracy={df_p2.loc[best_p2_idx, 'Accuracy']:.4f})")

## 10. Phase 3: VectorDB Comparison (SN 11-13)

**Variable**: 3 vector databases  
**Fixed**: Chunking=best from P1, Embedding=best from P2, LLM=LLaMA 3.3 70B, RAG=Naive

In [ ]:
# Phase 3: VectorDB Comparison
phase3_configs = [
    (11, "FAISS"),
    (12, "ChromaDB"),
    (13, "Qdrant"),
]

print(f"Phase 3: Testing 3 vector databases")
print(f"Fixed: Chunking={BEST_CHUNKING}, Embedding={BEST_EMBEDDING}, LLM=LLaMA 3.3 70B\n")

for sn, vdb_name in phase3_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=BEST_CHUNKING,
        embedding_model_name=BEST_EMBEDDING,
        vectordb_name=vdb_name,
        llm_model_name="LLaMA 3.3 70B",
        rag_type="Naive RAG",
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

# Visualize & select best
p3_data = [{"SN": sn, "VectorDB": vdb, **all_results[sn]["metrics"]} for sn, vdb in phase3_configs if sn in all_results]
df_p3 = pd.DataFrame(p3_data)
print("\nPhase 3 Results:")
print(df_p3[["SN", "VectorDB", "Accuracy", "F1", "Recall@5", "MRR"]].to_string(index=False))

best_p3_idx = df_p3["Accuracy"].idxmax()
BEST_VECTORDB = df_p3.loc[best_p3_idx, "VectorDB"]
print(f"\n>>> BEST VECTORDB: {BEST_VECTORDB} (Accuracy={df_p3.loc[best_p3_idx, 'Accuracy']:.4f})")

## 11. Phase 4: LLM Comparison (SN 14-16)

**Variable**: 3 LLM models  
**Fixed**: Chunking=best from P1, Embedding=best from P2, VectorDB=best from P3, RAG=Naive

In [ ]:
# Phase 4: LLM Comparison
phase4_configs = [
    (14, "LLaMA 3.3 70B"),
    (15, "LLaMA 3.1 8B"),
    (16, "Mixtral 8x7B"),
]

print(f"Phase 4: Testing 3 LLM models")
print(f"Fixed: Chunking={BEST_CHUNKING}, Embedding={BEST_EMBEDDING}, VectorDB={BEST_VECTORDB}\n")

for sn, llm_name in phase4_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=BEST_CHUNKING,
        embedding_model_name=BEST_EMBEDDING,
        vectordb_name=BEST_VECTORDB,
        llm_model_name=llm_name,
        rag_type="Naive RAG",
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

# Visualize & select best
p4_data = [{"SN": sn, "LLM": llm, **all_results[sn]["metrics"]} for sn, llm in phase4_configs if sn in all_results]
df_p4 = pd.DataFrame(p4_data)
print("\nPhase 4 Results:")
print(df_p4[["SN", "LLM", "Accuracy", "F1", "Recall@5", "MRR"]].to_string(index=False))

best_p4_idx = df_p4["Accuracy"].idxmax()
BEST_LLM = df_p4.loc[best_p4_idx, "LLM"]
print(f"\n>>> BEST LLM: {BEST_LLM} (Accuracy={df_p4.loc[best_p4_idx, 'Accuracy']:.4f})")

## 12. Phase 5: Final RAG Architecture Comparison (SN 17-20)

**Variable**: 4 RAG architectures  
**Fixed**: All components = best from Phases 1-4

**This is the core comparison of the thesis.**

In [ ]:
# Phase 5: Final RAG Architecture Comparison
phase5_configs = [
    (17, "Naive RAG"),
    (18, "Sparse RAG"),
    (19, "Hybrid RAG"),
    (20, "Multi-Hop RAG"),
]

print(f"Phase 5: Testing 4 RAG architectures")
print(f"Best config: Chunking={BEST_CHUNKING}, Embedding={BEST_EMBEDDING}, VectorDB={BEST_VECTORDB}, LLM={BEST_LLM}\n")

for sn, rag_type in phase5_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=BEST_CHUNKING,
        embedding_model_name=BEST_EMBEDDING,
        vectordb_name=BEST_VECTORDB,
        llm_model_name=BEST_LLM,
        rag_type=rag_type,
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

# Visualize
p5_data = [{"SN": sn, "RAG": rag, **all_results[sn]["metrics"]} for sn, rag in phase5_configs if sn in all_results]
df_p5 = pd.DataFrame(p5_data)
print("\nPhase 5 Results:")
print(df_p5[["SN", "RAG", "Accuracy", "Precision", "Recall", "F1", "Recall@5", "MRR"]].to_string(index=False))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, metric in zip(axes, ["Accuracy", "F1", "Recall@5", "MRR"]):
    bars = ax.bar(df_p5["RAG"], df_p5[metric], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    for bar, val in zip(bars, df_p5[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", fontsize=9)

plt.suptitle("Phase 5: RAG Architecture Comparison (Best Config)", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "phase5_rag_comparison.png", bbox_inches="tight", dpi=150)
plt.show()

## 13. Phase 6: Cross-Validation Experiments (SN 21-29)

Test whether the optimal chunking strategy varies by RAG architecture.

In [ ]:
# Phase 6: Cross-Validation Experiments
phase6_configs = [
    (21, "Sparse RAG",    "Semantic Passage (200 tokens)"),
    (22, "Sparse RAG",    "Sentence Window (4 sentences, overlap=1)"),
    (23, "Sparse RAG",    "Overlapping (512 tokens, 50 overlap)"),
    (24, "Hybrid RAG",    "Semantic Passage (200 tokens)"),
    (25, "Hybrid RAG",    "Sentence Window (4 sentences, overlap=1)"),
    (26, "Hybrid RAG",    "Overlapping (512 tokens, 50 overlap)"),
    (27, "Multi-Hop RAG", "Semantic Passage (200 tokens)"),
    (28, "Multi-Hop RAG", "Sentence Window (4 sentences, overlap=1)"),
    (29, "Multi-Hop RAG", "Overlapping (512 tokens, 50 overlap)"),
]

print(f"Phase 6: Cross-validation — 9 experiments")
print(f"Fixed: Embedding={BEST_EMBEDDING}, VectorDB={BEST_VECTORDB}, LLM={BEST_LLM}\n")

for sn, rag_type, chunking_name in phase6_configs:
    metrics, details = run_experiment(
        df_questions=df_sample,
        textbook_corpus=textbook_corpus,
        chunking_name=chunking_name,
        embedding_model_name=BEST_EMBEDDING,
        vectordb_name=BEST_VECTORDB,
        llm_model_name=BEST_LLM,
        rag_type=rag_type,
        groq_client=groq_client,
    )
    all_results[sn] = {"metrics": metrics, "details": details}
    write_metrics_to_excel(EXCEL_PATH, sn, metrics)

# Visualize
p6_data = [{"SN": sn, "RAG": rag, "Chunking": chunk, **all_results[sn]["metrics"]}
           for sn, rag, chunk in phase6_configs if sn in all_results]
df_p6 = pd.DataFrame(p6_data)
print("\nPhase 6 Results:")
print(df_p6[["SN", "RAG", "Chunking", "Accuracy", "F1", "MRR"]].to_string(index=False))

print("\nPhase 6 complete!")

## 14. Final Summary & Best Configuration

In [ ]:
print("=" * 60)
print("COMPONENT SELECTION EXPERIMENTS — FINAL SUMMARY")
print("=" * 60)
print(f"\n  Best Chunking Strategy : {BEST_CHUNKING}")
print(f"  Best Embedding Model   : {BEST_EMBEDDING}")
print(f"  Best Vector Database   : {BEST_VECTORDB}")
print(f"  Best LLM Model         : {BEST_LLM}")
print(f"\n  Total experiments run  : {len(all_results)}")
print(f"  Questions per experiment: {SAMPLE_SIZE}")
print(f"\n  Results saved to: {EXCEL_PATH.resolve()}")
print("=" * 60)

# Export summary CSV
summary_rows = []
for sn in sorted(all_results.keys()):
    row = {"SN": sn}
    row.update(all_results[sn]["metrics"])
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "component_selection_results.csv"
df_summary.to_csv(summary_path, index=False)
print(f"\nCSV exported to: {summary_path.resolve()}")

print("\n>>> UPDATE config/config.py with these selections before running Notebook 03 <<<")
print(f"""
SELECTED_CHUNKING = "{BEST_CHUNKING}"
SELECTED_EMBEDDING_MODEL = "{BEST_EMBEDDING}"
SELECTED_VECTORDB = "{BEST_VECTORDB}"
SELECTED_LLM_MODEL = "{LLM_MODEL_MAP.get(BEST_LLM, BEST_LLM)}"
""")